# JSE Sector Momentum Strategy - Research Notebook
## Stage 4: Signal Construction

### Objective
Construct a 12-1 month momentum signal and use it to create a table that ranks each of the sectors every month.

### Input File
data/jse_sector_monthly_log_returns_2019_2026.csv

### Steps
1. Load monthly log returns from Stage 3
2. Calculate the 12-1 month momentum score for each sector each month
3. Rank the sectors each month from highest to lowest momentum score
4. Output a rankings table showing which sector ranked 1st through 4th every month and store it

### Key Findings
During the COVID period all sectors showed negative momentum scores simultaneously. For simplicity the signal follows the rule of always investing in the highest ranked sector, assuming that in times of general decline the most resilient sector offers the best opportunity. A cash filter that moves to cash when all signals are negative is identified as a potential improvement for future research.

### Output
- data/jse_sector_momentum_signal.csv — the sector to invest in each month
- data/jse_sector_momentum_signal_rankings.csv — full rankings table showing rank 1 through 4 for each sector every month

### Next Step
Stage 5 — Backtesting: Use the signal files to simulate the trading strategy and measure performance against the ALSI benchmark.

# JSE Sector Momentum Strategy - Research Notebook
## Stage 4: Signal Construction

### Objective
Construct a 12-1 month momentum signal and use it to create a table 
that ranks each of the sectors every month.

### Input File
data/jse_sector_prices_clean_2019_2026.csv

### Steps
1. Load clean daily prices and convert to monthly log returns
2. Calculate the 12-1 month momentum score for each sector each month
3. Rank the sectors each month from highest to lowest momentum score
4. Output a rankings table showing which sector ranked 1st through 4th every month


## Step 1: Load Clean Daily Prices & Convert to Monthly Log Returns

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

monthly_returns_df = pd.read_csv("data/jse_sector_monthly_log_returns_2019_2026.csv", index_col=0, parse_dates=True)
print(monthly_returns_df.head())

            Financials  Resources  Industrials  Consumer Goods
Date                                                          
2019-02-28   -0.021239   0.087303    -0.024094        0.118959
2019-03-31   -0.046531   0.030868    -0.058845        0.014972
2019-04-30    0.044953  -0.023175     0.057251       -0.006261
2019-05-31   -0.025558  -0.052975    -0.039682       -0.021071
2019-06-30    0.011440   0.096746    -0.044402        0.076821


## Step 2 Calculate 12-1 Momentum Score

For each month, sum the xcvbrns from 12 months ago to 2 months ago, skippong the most recent month. This gives an 11 month return window with 1-month skip.

In [2]:
# Calculate the 11 month momentum for each sector, shift it by 1 month to avoid look-ahead bias, and drop the NaN values that result from the rolling calculation
momentum = monthly_returns_df.rolling(11).sum().shift(1).dropna()
print(momentum.head())
print(momentum.shape)

            Financials  Resources  Industrials  Consumer Goods
Date                                                          
2020-01-31   -0.106109   0.179122    -0.152173        0.161076
2020-02-29   -0.138881   0.056357    -0.191020        0.055290
2020-03-31   -0.192067  -0.097614    -0.288663       -0.031395
2020-04-30   -0.589625  -0.225399    -0.614482       -0.075140
2020-05-31   -0.481679   0.033712    -0.520474        0.036434
(76, 4)


### Step 3 — Rank Sectors by Momentum Score Each Month

Rank sectors from 1 (highest momentum) to 4 (lowest momentum) each month. Rank 1 is the sector we will invest in during Stage 5.

In [3]:
signal_rankings = momentum.rank(axis=1, ascending=False)
print(signal_rankings.head())

            Financials  Resources  Industrials  Consumer Goods
Date                                                          
2020-01-31         3.0        1.0          4.0             2.0
2020-02-29         3.0        1.0          4.0             2.0
2020-03-31         3.0        2.0          4.0             1.0
2020-04-30         3.0        2.0          4.0             1.0
2020-05-31         3.0        2.0          4.0             1.0


### Step 4 — Extract Top Ranked Sector Each Month

Identify the sector ranked 1st each month. This becomes the investment signal consumed by the backtest in Stage 5.

In [4]:
signal = signal_rankings.idxmin(axis=1)
print(signal.head())

Date
2020-01-31         Resources
2020-02-29         Resources
2020-03-31    Consumer Goods
2020-04-30    Consumer Goods
2020-05-31    Consumer Goods
dtype: str


### Step 4 — Save the signal and the signal rankings for use in Stage 5

In [5]:
signal_rankings.to_csv("data/jse_sector_momentum_signal_rankings.csv")
signal.to_csv("data/jse_sector_momentum_signal.csv")
print("Saved signal rankings and signal for use in Stage 5")

Saved signal rankings and signal for use in Stage 5
